# 03 — OCR: PaddleOCR (zero-shot)

Off-the-shelf PaddleOCR with built-in angle classifier (`use_angle_cls=True`).
No training — direct inference on pre-computed crops.
Handles 0°/180° rotation automatically via built-in angle classifier.
Two crop paths compared: `wm_polygon`, `wm_bbox`.

Requires: `pip install paddlepaddle paddleocr` (CPU) or `pip install paddlepaddle-gpu paddleocr` (GPU).

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"], check=True)
    BRANCH = "feature/ocr-notebooks"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rapidfuzz", "shapely", "paddlepaddle", "paddleocr"], check=True)
    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..")
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json, time, csv
import cv2, numpy as np
from datetime import datetime
from tqdm import tqdm

print(f"ROOT: {ROOT}")

In [ ]:
CROPS_ROOT = DATA_ROOT / "ocr_crops"
WM_PATH    = DATA_ROOT / "waterMeterDataset/WaterMeters"

RUN_NAME = f"paddle_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Run: {RUN_NAME}")

In [ ]:
from paddleocr import PaddleOCR

from models.data.ocr_dataset import prepare_ocr_crops, load_ocr_crops
from models.metrics.evaluation import full_string_accuracy, per_digit_accuracy, character_error_rate

prepare_ocr_crops(WM_PATH, CROPS_ROOT)
print("Crops ready.")

# use_angle_cls=True — built-in 0/180 angle classifier
ocr_engine = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
print("PaddleOCR ready.")

In [ ]:
def run_paddle_ocr(engine, img_path) -> tuple[str, float]:
    """Run PaddleOCR on a pre-cropped image, return (digit_string, avg_confidence).

    PaddleOCR v4: cls is configured at init (use_angle_cls=True), not at call time.
    """
    result = engine.ocr(str(img_path))  # no cls= kwarg in v4
    if not result:
        return "", 0.0
    lines = result[0] if isinstance(result[0], list) else result
    if not lines:
        return "", 0.0

    items = []  # (x_left, digits, confidence)
    for entry in lines:
        if entry is None:
            continue
        box, (text, score) = entry
        digits = "".join(c for c in text if c.isdigit())
        if digits:
            x_left = min(pt[0] for pt in box)
            items.append((x_left, digits, float(score)))

    if not items:
        return "", 0.0

    items.sort(key=lambda x: x[0])
    pred = "".join(it[1] for it in items)
    avg_conf = float(np.mean([it[2] for it in items]))
    return pred, avg_conf


def eval_paddle(engine, samples):
    preds, gts, times, confs = [], [], [], []
    for img_path, label_gt in tqdm(samples):
        t0 = time.perf_counter()
        pred, conf = run_paddle_ocr(engine, img_path)
        times.append((time.perf_counter() - t0) * 1000)
        preds.append(pred)
        gts.append(label_gt)
        confs.append(conf)

    fsa_raw  = full_string_accuracy(preds, gts)
    fsa_norm = full_string_accuracy([p.lstrip("0") or "0" for p in preds],
                                    [g.lstrip("0") or "0" for g in gts])
    pda = float(np.mean([per_digit_accuracy(p, g) for p, g in zip(preds, gts)]))
    cer = float(np.mean([character_error_rate(p, g) for p, g in zip(preds, gts)]))
    ms  = float(np.mean(times))
    avg_conf = float(np.mean(confs)) if confs else 0.0
    return fsa_raw, fsa_norm, pda, cer, ms, avg_conf, preds, gts


print("Helpers loaded.")

In [ ]:
wm_poly_test = load_ocr_crops(CROPS_ROOT / "wm_polygon", "test")
print(f"WM polygon test={len(wm_poly_test)}")

wm_poly_fsa_raw, wm_poly_fsa_norm, wm_poly_pda, wm_poly_cer, wm_poly_ms, wm_poly_conf, wm_poly_preds, wm_poly_gts = eval_paddle(ocr_engine, wm_poly_test)
print(f"WM poly — FSA raw={wm_poly_fsa_raw:.3f} norm={wm_poly_fsa_norm:.3f} PDA={wm_poly_pda:.3f} CER={wm_poly_cer:.3f} conf={wm_poly_conf:.3f} {wm_poly_ms:.1f}ms")

In [ ]:
wm_bbox_test = load_ocr_crops(CROPS_ROOT / "wm_bbox", "test")
print(f"WM bbox test={len(wm_bbox_test)}")

wm_bbox_fsa_raw, wm_bbox_fsa_norm, wm_bbox_pda, wm_bbox_cer, wm_bbox_ms, wm_bbox_conf, wm_bbox_preds, wm_bbox_gts = eval_paddle(ocr_engine, wm_bbox_test)
print(f"WM bbox — FSA raw={wm_bbox_fsa_raw:.3f} norm={wm_bbox_fsa_norm:.3f} PDA={wm_bbox_pda:.3f} CER={wm_bbox_cer:.3f} conf={wm_bbox_conf:.3f} {wm_bbox_ms:.1f}ms")

In [ ]:
print(f"{'='*60}")
print(f"{'Metric':<20} {'WM polygon':>18} {'WM bbox':>18}")
print(f"{'='*60}")
for name, poly_v, bbox_v in [
    ("FSA raw",   wm_poly_fsa_raw,  wm_bbox_fsa_raw),
    ("FSA norm",  wm_poly_fsa_norm, wm_bbox_fsa_norm),
    ("Per-digit", wm_poly_pda,      wm_bbox_pda),
    ("CER",       wm_poly_cer,      wm_bbox_cer),
    ("Avg conf",  wm_poly_conf,     wm_bbox_conf),
]:
    print(f"{name:<20} {poly_v:>18.3f} {bbox_v:>18.3f}")
print(f"{'Inference ms':<20} {wm_poly_ms:>18.1f} {wm_bbox_ms:>18.1f}")
print(f"{'='*60}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
rows = [
    ("WM polygon", CROPS_ROOT / "wm_polygon", wm_poly_preds, wm_poly_gts),
    ("WM bbox",    CROPS_ROOT / "wm_bbox",    wm_bbox_preds, wm_bbox_gts),
]
for row_i, (label, crops_dir, preds, gts) in enumerate(rows):
    samples = load_ocr_crops(crops_dir, "test")
    for col_i, ax in enumerate(axes[row_i]):
        if col_i >= len(samples):
            ax.axis("off"); continue
        img = cv2.imread(str(samples[col_i][0]))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        gt  = gts[col_i]  if col_i < len(gts)   else "?"
        pr  = preds[col_i] if col_i < len(preds) else "?"
        color = "green" if pr == gt else "red"
        ax.set_title(f"GT={gt}\nPred={pr}", fontsize=9, color=color)
        ax.axis("off")
    axes[row_i][0].set_ylabel(label, fontsize=11)
plt.suptitle("PaddleOCR Predictions", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ocr_paddle_predictions.png", dpi=150)
plt.close()
print("Saved ocr_paddle_predictions.png")

In [ ]:
metrics = {
    "method": "paddle_ocr",
    "wm_polygon": {
        "fsa_raw":          round(wm_poly_fsa_raw,  4),
        "fsa_norm":         round(wm_poly_fsa_norm, 4),
        "per_digit_acc":    round(wm_poly_pda,       4),
        "cer":              round(wm_poly_cer,        4),
        "avg_confidence":   round(wm_poly_conf,       4),
        "avg_inference_ms": round(wm_poly_ms,         1),
    },
    "wm_bbox": {
        "fsa_raw":          round(wm_bbox_fsa_raw,  4),
        "fsa_norm":         round(wm_bbox_fsa_norm, 4),
        "per_digit_acc":    round(wm_bbox_pda,       4),
        "cer":              round(wm_bbox_cer,        4),
        "avg_confidence":   round(wm_bbox_conf,       4),
        "avg_inference_ms": round(wm_bbox_ms,         1),
    },
    "config": {
        "use_angle_cls": True,
        "lang": "en",
        "training": "zero-shot",
    },
    "run_date": datetime.now().isoformat(),
}
with open(RESULTS_DIR / "ocr_paddle_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "ocr_comparison.csv"
header = ["method",
          "wm_poly_fsa_raw", "wm_poly_fsa_norm", "wm_poly_pda", "wm_poly_cer", "wm_poly_ms",
          "wm_bbox_fsa_raw", "wm_bbox_fsa_norm", "wm_bbox_pda", "wm_bbox_cer", "wm_bbox_ms",
          "run_date"]
write_hdr = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    w = csv.writer(f)
    if write_hdr: w.writerow(header)
    w.writerow(["paddle_ocr",
                round(wm_poly_fsa_raw,4),  round(wm_poly_fsa_norm,4),
                round(wm_poly_pda,4),       round(wm_poly_cer,4),       round(wm_poly_ms,1),
                round(wm_bbox_fsa_raw,4),  round(wm_bbox_fsa_norm,4),
                round(wm_bbox_pda,4),       round(wm_bbox_cer,4),       round(wm_bbox_ms,1),
                datetime.now().strftime("%Y-%m-%d %H:%M")])
print("Results saved.")

## Conclusions

- **WM polygon:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **WM bbox:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **Key advantage:** Zero-shot — no training required. Built-in `use_angle_cls=True` handles 0°/180° rotation automatically.
- **Key limitation:** Not fine-tuned on water meter crops; general-purpose recognition may misread noisy/stylized meter fonts.
- **Key finding:** fill after running.